<a href="https://colab.research.google.com/github/mkvkanpur/hpc/blob/main/struct_warp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
try:
    import warp as wp
    print(f"Warp version {wp.__version__} is ready!")
except ImportError:
    print("Warp not found. Installing...")
    !pip install warp-lang
    import warp as wp
    wp.init()

# Check if CUDA is actually available on this specific machine
if wp.is_cuda_available():
    device = "cuda"
else:
    device = "cpu"
    print("Warning: CUDA not found. Falling back to CPU.")

print(f"Using device: {device}")

Warp version 1.12.0 is ready!
Using device: cuda


In [58]:
import warp as wp
import torch
import numpy as np

wp.init()

N = 64
Nx = N
Ny = N
STR_SIZE = int(np.sqrt(Nx**2 + Ny**2)/2)
TILE_x = wp.constant(8)
TILE_y = wp.constant(8)
NO_TILES_x = (Nx//(2*TILE_x))
NO_TILES_y = (Ny//(2*TILE_y))


@wp.kernel
def compute_str(
    ux: wp.array2d(dtype=float),
    uy: wp.array2d(dtype=float),
    str_tiled: wp.array3d(dtype=float),
    l_min: wp.vec2, l_max: wp.vec2
):
    i, j = wp.tid()

    ux_tile = wp.tile_load(ux, shape=(TILE_x, TILE_y), offset=(i*TILE_x, j*TILE_y))
    uy_tile = wp.tile_load(uy, shape=(TILE_x, TILE_y), offset=(i*TILE_x, j*TILE_y))

    for lx in range(int(l_min[0]), int(l_max[0])):
      for ly in range(int(l_min[1]), int(l_max[1])):

        ux_tile_shifted = wp.tile_load(ux, shape=(TILE_x, TILE_y), offset=(i*TILE_x+lx, j*TILE_y+ly))
        uy_tile_shifted = wp.tile_load(uy, shape=(TILE_x, TILE_y), offset=(i*TILE_x+lx, j*TILE_y+ly))

        dux_tile = ux_tile_shifted - ux_tile
        duy_tile = uy_tile_shifted - uy_tile

        l_vec = wp.vec2(float(lx), float(ly))
        l_mag = wp.length(l_vec)

        # 2. Now the math will work because these are tiles of floats
        #l_mag = wp.length(wp.vec2(float(lx), float(ly)))

        # Calculate the base projection value for each element in the tile
        base_projection_tile = (dux_tile * float(lx) + duy_tile * float(ly))/l_mag

        # Manually compute power (cubed) for each element in the tile
        # `base_projection_tile` is assumed to be mutable for in-place modification
        for x in range(TILE_x):
            for y in range(TILE_y):
                base_projection_tile[x, y] = wp.pow(base_projection_tile[x, y], 3.0)

        # The 'projections' variable now holds the cubed results
        projections_cubed = base_projection_tile

        # 3. Sum the cubed results across the tile
        tile_sum = wp.tile_sum(projections_cubed) # Summing the cubed projections

        num_elements = float(TILE_x * TILE_y)

        partial_mean = tile_sum[0]/ num_elements
        str_index = int(l_mag)
        str_tiled[str_index, i, j] = partial_mean


### MAIN ####
ux_np = np.arange(N, dtype=np.float32).reshape(-1, 1) * np.ones((1, N), dtype=np.float32)
uy_np = np.arange(N, dtype=np.float32).reshape(1, -1) * np.ones((N, 1), dtype=np.float32)

print(ux_np[2,2], uy_np[2,2])

ux_wp = wp.array(ux_np, dtype=float, device="cuda")
uy_wp = wp.array(uy_np, dtype=float, device="cuda")

str_tiled = wp.zeros((STR_SIZE,NO_TILES_x, NO_TILES_y), dtype=float)
str = wp.zeros(STR_SIZE, dtype=float)

l_min = wp.vec2(2.0, 2.0)
l_max = wp.vec2(float(Nx//3), float(Ny//3))
# # Corrected block_dim from TILE_THREADS to (TILE_x, TILE_y)
wp.launch_tiled(compute_str, dim=[NO_TILES_x, NO_TILES_y], inputs=[ux_wp, uy_wp, str_tiled, l_min, l_max], block_dim=128)
wp.synchronize()

# Convert wp.array to torch.Tensor before calling torch.sum
str_tiled_torch = wp.to_torch(str_tiled)
str = torch.sum(str_tiled_torch, dim=(1, 2))/(NO_TILES_x*NO_TILES_y)
print(f"str = {str}")

2.0 2.0
Module __main__ 362cfc0 load on device 'cuda:0' took 7506.82 ms  (compiled)
str = tensor([0.0000e+00, 0.0000e+00, 2.2627e+01, 4.6872e+01, 8.9443e+01, 1.9825e+02,
        3.0187e+02, 4.4171e+02, 7.1554e+02, 9.5534e+02, 1.2494e+03, 1.6035e+03,
        2.0239e+03, 2.7021e+03, 3.2854e+03, 3.9528e+03, 4.7104e+03, 5.5641e+03,
        6.8305e+03, 7.9102e+03, 9.1039e+03, 1.0549e+04, 1.1892e+04, 1.3573e+04,
        1.4550e+04, 1.6802e+04, 1.9481e+04, 2.0993e+04, 2.2627e+04, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00], device='cuda:0')


In [ ]:
import warp as wp
import torch
import numpy as np

wp.init()
device = "cuda" # Or "cpu" / "mps" for your Mac Neo

@wp.kernel
def compute_structure_function(
    u_field: wp.array2d(dtype=wp.vec2),
    str_results: wp.array(dtype=float),
    l_min, l_max
):


  u_quad = u_field[0:limit, 0:limit]

  for lx in range(l_min[0], lmax[0]):
    for ly in range(l_min[1], lmax[1]):
      up_quad = u_field[lx:limit+lx, ly:limit+ly]
      diff = up_quad-u_quad


# --- Setup and Launch ---

# 1. Create dummy PyTorch data (Velocity field)
N = 10

#ux = torch.randn((N, N), device=device)
#uy = torch.randn((N, N), device=device)

ux = np.arange(N, dtype=np.float32).reshape(-1, 1) * np.ones((1, N), dtype=np.float32)
uy = np.arange(N, dtype=np.float32).reshape(1, -1) * np.ones((N, 1), dtype=np.float32)
str = np.zeros(N, dtype=np.float32)

ux_t = torch.from_numpy(ux).to(device)
uy_t = torch.from_numpy(uy).to(device)
str_t = torch.from_numpy(str).to(device)

# 2. Combine into a vec2 field and "Wrap" it
# Shape: (limit, limit, 2)
u_combined = torch.stack([ux, uy], dim=-1).contiguous()
u_wp = wp.from_torch(u_combined, dtype=wp.vec2)

l_min = [1,1]
l_max = [4,4]

# 4. Prepare output array
str_out_wp = wp.zeros(N, dtype=float, device=device)

# wp.launch(
#     kernel=compute_structure_function,
#     dim=len(l_data), # One thread per l-vector
#     inputs=[u_wp, str_out_wp, l_min, l_max],
#     device=device
# )

# # 6. Bring back to PyTorch for your manuscript
# str_final = wp.to_torch(str_out_wp)
# print("Structure Function Results:", str_final)

WarpCodegenError: Incomplete argument annotations on function compute_structure_function

In [ ]:
N = 10

ux = np.arange(N, dtype=np.float32).reshape(-1, 1) * np.ones((1, N), dtype=np.float32)
uy = np.arange(N, dtype=np.float32).reshape(1, -1) * np.ones((N, 1), dtype=np.float32)
str = np.zeros(N, dtype=np.float32)

ux_t = torch.from_numpy(ux).to(device)
uy_t = torch.from_numpy(uy).to(device)
str_t = torch.from_numpy(str).to(device)

# 2. Combine into a vec2 field and "Wrap" it
# Shape: (limit, limit, 2)
u_combined = torch.stack([ux_t, uy_t], dim=-1).contiguous()
u_wp = wp.from_torch(u_combined, dtype=wp.vec2)

l_min = [1,1]
l_max = [4,4]

up_wp = u_wp[0:3, 0:2]
print(up_wp)

[[[0. 0.]
  [0. 1.]]

 [[1. 0.]
  [1. 1.]]

 [[2. 0.]
  [2. 1.]]]
